# The Backdoor Criterion and Good vs. Bad Controls

## Learning Objectives

By the end of this notebook you will be able to:

1. State the **adjustment problem**: given a causal DAG, which variables should we condition on to estimate the causal effect of $X$ on $Y$?
2. Define and apply the **Backdoor Criterion** (Pearl) to identify valid adjustment sets.
3. Derive the **backdoor adjustment formula** $P(Y \mid \text{do}(X)) = \sum_s P(Y \mid X, S{=}s)\,P(S{=}s)$.
4. Identify **good controls** (confounders, precision variables) and explain why they reduce bias.
5. Identify **four types of bad controls** (post-treatment, collider, precision parasite, bias amplifier) and demonstrate through simulation how each introduces bias.
6. Explain the **Table 2 Fallacy** and why interpreting all regression coefficients causally is wrong.
7. Use **DoWhy** to automate identification and estimation of causal effects from a DAG specification.

## Prerequisites

- [Module 06 — Linear regression](../06_linear_models/01_simple_linear_regression.ipynb)
- [Module 10.01 — Introduction to causal inference and DAGs](01_intro_causal_dags.ipynb)
- [Module 10.02 — d-Separation and conditional independence](02_d_separation.ipynb)

In [ ]:
import sys, os, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        "pip install -q networkx dowhy statsmodels"
    )

_miktex_bin = Path.home() / "AppData/Local/Programs/MiKTeX/miktex/bin/x64"
if _miktex_bin.exists() and str(_miktex_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + str(_miktex_bin)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import statsmodels.api as sm
from scipy import stats

sys.path.insert(0, os.path.abspath("../../src"))
from amstats.plotting import apply_style, SALMON, GOLD, EMERALD, CYAN, PERIWINKLE, ORCHID

apply_style()

# Optional: DoWhy for automated causal inference
try:
    import dowhy
    from dowhy import CausalModel
    HAS_DOWHY = True
    print(f"DoWhy {dowhy.__version__} available.")
except ImportError:
    HAS_DOWHY = False
    print("DoWhy not installed — Section 7 will be skipped.")


class Cfg:
    root = Path("../../").resolve()
    gif_dir = root / "media" / "gifs"
    has_latex: bool = (
        shutil.which("latex") is not None or shutil.which("pdflatex") is not None
    )

    def __init__(self):
        self.gif_dir.mkdir(parents=True, exist_ok=True)

    def apply_manim_config(self):
        from manim import config as mcfg
        mcfg.format = "gif"

    def math_text(self, expr, **kwargs):
        from manim import MathTex, Text
        if self.has_latex:
            return MathTex(expr, **kwargs)
        return Text(expr, **kwargs)

    def save_gifs(self, clean=True):
        local_media = Path("media")
        found = list(local_media.rglob("*.gif")) if local_media.exists() else []
        if not found:
            print("  No new GIFs to save.")
            return
        for gif in found:
            dest = self.gif_dir / gif.name
            shutil.copy2(gif, dest)
            print(f"  \u2713 media/gifs/{gif.name}")
        if clean:
            for sub in ("videos", "images", "Tex"):
                d = local_media / sub
                if d.exists():
                    shutil.rmtree(d, ignore_errors=True)
            print("  Cleaned up local temp render files (kept media/jupyter/).")


cfg = Cfg()
rng = np.random.default_rng(42)


# ── DAG drawing helper ──
def draw_dag(G, pos, ax=None, title="", node_color=PERIWINKLE,
             highlight_nodes=None, highlight_color=SALMON,
             edge_style="->", figsize=(5, 3.5)):
    """Draw a DAG with consistent styling."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()

    colors = []
    for node in G.nodes():
        if highlight_nodes and node in highlight_nodes:
            colors.append(highlight_color)
        else:
            colors.append(node_color)

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors,
                           node_size=1200, edgecolors="black", linewidths=1.5)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=14, font_weight="bold")
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="black",
                           arrowsize=20, arrowstyle="->",
                           connectionstyle="arc3,rad=0.1",
                           width=2, min_source_margin=20, min_target_margin=20)
    ax.set_title(title, fontsize=13)
    ax.axis("off")
    return fig, ax

---

## 1. The Adjustment Problem

### The fundamental question

Suppose we want to estimate the **causal effect** of a treatment $X$ on an outcome $Y$.  We have observational data (no randomised experiment) and a causal DAG that encodes our assumptions about the data-generating process.

The naive approach is to estimate $E[Y \mid X = x]$ — the conditional expectation.  But in the presence of **confounders** (common causes of $X$ and $Y$), the conditional expectation conflates the causal effect with spurious associations through backdoor paths.

The standard remedy is **statistical adjustment**: condition on (i.e., control for) some set of variables $\mathbf{S}$ to block the confounding paths.  This raises the central question:

> **Which variables should we include in $\mathbf{S}$?**

This question seems simple, but the answer is subtle.  Getting it wrong — adjusting for the **wrong** variables — can:

- **Introduce bias** where none existed (collider bias)
- **Amplify existing bias** from unmeasured confounders
- **Remove part of the causal effect** we are trying to estimate (post-treatment bias)
- **Reduce precision** without reducing confounding

The "kitchen sink" approach of throwing every available variable into a regression is not just unhelpful — it can be actively harmful.

### Notation

Throughout this notebook:

| Symbol | Meaning |
|--------|---------|
| $X$ | Treatment / exposure (the cause we are studying) |
| $Y$ | Outcome |
| $\text{do}(X{=}x)$ | An intervention that sets $X$ to the value $x$, regardless of its natural causes |
| $P(Y \mid \text{do}(X{=}x))$ | The **causal** (interventional) distribution of $Y$ when we force $X{=}x$ |
| $P(Y \mid X{=}x)$ | The **observational** (conditional) distribution — may include confounding |
| $\mathbf{S}$ | A set of variables we condition on (the "adjustment set") |

### A motivating example

Consider a study of whether a drug ($X$) reduces blood pressure ($Y$).  Older patients are more likely to receive the drug *and* more likely to have high blood pressure.  Age ($Z$) is a **confounder**.

$$Z \to X, \quad Z \to Y, \quad X \to Y$$

If we simply compare treated vs. untreated patients, the drug may *appear* less effective (or even harmful) because treated patients are older and sicker.  We need to **adjust for age** to isolate the causal effect of the drug.

But what if we also observe a variable $M$ that is a *mediator* — the drug works by lowering cholesterol ($M$), which then lowers blood pressure?  $X \to M \to Y$.  Should we adjust for $M$?  **No** — conditioning on a mediator would block part of the causal pathway, understating the total effect of the drug.

The Backdoor Criterion gives us a principled, graph-based answer to these questions.

In [ ]:
# Motivating example: confounder vs. mediator
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# DAG 1: Confounder Z
G1 = nx.DiGraph()
G1.add_edges_from([("Z (Age)", "X (Drug)"), ("Z (Age)", "Y (BP)"), ("X (Drug)", "Y (BP)")])
pos1 = {"Z (Age)": (1, 1), "X (Drug)": (0, 0), "Y (BP)": (2, 0)}
draw_dag(G1, pos1, ax=axes[0], title="Confounder: adjust for Z",
         highlight_nodes={"Z (Age)"}, highlight_color=EMERALD)

# DAG 2: Mediator M
G2 = nx.DiGraph()
G2.add_edges_from([("X (Drug)", "M (Chol)"), ("M (Chol)", "Y (BP)"), ("X (Drug)", "Y (BP)")])
pos2 = {"X (Drug)": (0, 0), "M (Chol)": (1, 1), "Y (BP)": (2, 0)}
draw_dag(G2, pos2, ax=axes[1], title="Mediator: do NOT adjust for M",
         highlight_nodes={"M (Chol)"}, highlight_color=SALMON)

plt.tight_layout()
plt.show()

---

## 2. The Backdoor Criterion

### Definition (Pearl, 1993)

A set of variables $\mathbf{S}$ satisfies the **backdoor criterion** relative to an ordered pair of variables $(X, Y)$ in a DAG $\mathcal{G}$ if:

**(i)** No node in $\mathbf{S}$ is a **descendant** of $X$.

**(ii)** $\mathbf{S}$ **blocks every path** between $X$ and $Y$ that contains an arrow **into** $X$ (i.e., every "backdoor path").

A **backdoor path** from $X$ to $Y$ is any path that starts with an arrow *into* $X$ (i.e., $\leftarrow$ at $X$).  These are non-causal paths — they transmit spurious association but not causal influence.

### The Backdoor Adjustment Formula

**Theorem.** If a set $\mathbf{S}$ satisfies the backdoor criterion relative to $(X, Y)$, then the causal effect of $X$ on $Y$ is identifiable and given by:

$$\boxed{P(Y \mid \text{do}(X{=}x)) = \sum_{\mathbf{s}} P(Y \mid X{=}x, \mathbf{S}{=}\mathbf{s})\,P(\mathbf{S}{=}\mathbf{s})}$$

For continuous variables, the sum becomes an integral:

$$P(Y \mid \text{do}(X{=}x)) = \int P(Y \mid X{=}x, \mathbf{S}{=}\mathbf{s})\,f_{\mathbf{S}}(\mathbf{s})\,d\mathbf{s}$$

In the linear case with a single confounder $S$, this reduces to the familiar **regression adjustment**: the coefficient of $X$ in the regression $Y \sim X + S$ is the causal effect.

### Why does it work?

The intuition is as follows:

1. **Backdoor paths** create spurious (non-causal) associations between $X$ and $Y$.
2. **Condition (ii)** ensures that all such spurious paths are blocked by conditioning on $\mathbf{S}$.
3. **Condition (i)** ensures that conditioning on $\mathbf{S}$ does not accidentally block or distort causal paths from $X$ to $Y$ (which would happen if $\mathbf{S}$ contained descendants of $X$).

Together, these conditions guarantee that within each stratum of $\mathbf{S}$, the only remaining association between $X$ and $Y$ is causal.  Averaging over the strata (the marginalisation) gives us the causal effect in the population.

### Proof sketch

The formal proof uses the **truncated factorisation** (also called the **g-formula** or **manipulation theorem**).  Under the intervention $\text{do}(X{=}x)$, we delete all arrows into $X$ from the DAG and replace the structural equation for $X$ with the constant $x$.  The joint distribution of all other variables factors according to the **mutilated graph** $\mathcal{G}_{\overline{X}}$:

$$P(y \mid \text{do}(x)) = \sum_{\mathbf{s}} \prod_{V_i \neq X} P(v_i \mid \text{pa}_i)$$

where the product is taken over the conditional distributions in the mutilated graph.  When $\mathbf{S}$ satisfies the backdoor criterion, the above simplifies to the adjustment formula because the only relevant conditional distributions are $P(Y \mid X, \mathbf{S})$ and $P(\mathbf{S})$.

### Worked Example: Identifying the Adjustment Set

Consider the following DAG:

- $U$ is an unobserved confounder
- $Z$ is an observed common cause of $X$ and $Y$  
- $M$ is a mediator ($X \to M \to Y$)
- $C$ is a collider ($X \to C \leftarrow Y$)

We want the causal effect of $X$ on $Y$.

**Backdoor paths from $X$ to $Y$:**

1. $X \leftarrow Z \rightarrow Y$ — this is an open backdoor path.  Blocked by conditioning on $Z$.

**Valid adjustment sets:**

- $\{Z\}$ satisfies the backdoor criterion: $Z$ is not a descendant of $X$, and $\{Z\}$ blocks the only backdoor path.
- $\emptyset$ does **not** work — the backdoor path $X \leftarrow Z \rightarrow Y$ is open.
- $\{M\}$ violates condition (i) — $M$ is a descendant of $X$.
- $\{C\}$ violates condition (i) — $C$ is a descendant of $X$.  Worse, conditioning on the collider $C$ would **open** a new spurious path.

In [ ]:
# Worked example DAG
G_ex = nx.DiGraph()
G_ex.add_edges_from([
    ("Z", "X"), ("Z", "Y"),          # confounder
    ("X", "Y"),                        # direct causal effect
    ("X", "M"), ("M", "Y"),            # mediator
    ("X", "C"), ("Y", "C"),            # collider
])
pos_ex = {"Z": (1, 2), "X": (0, 1), "Y": (2, 1), "M": (1, 0), "C": (1, -1)}

fig, ax = plt.subplots(figsize=(6, 5))
draw_dag(G_ex, pos_ex, ax=ax,
         title="Worked Example: Identifying the Adjustment Set",
         highlight_nodes={"Z"}, highlight_color=EMERALD)

# Add annotation
ax.annotate("Confounder\n(adjust for this)", xy=(1, 2), xytext=(2.3, 2.3),
            fontsize=9, color=EMERALD, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=EMERALD, lw=1.5))
ax.annotate("Mediator\n(do NOT adjust)", xy=(1, 0), xytext=(2.3, 0),
            fontsize=9, color=SALMON, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=SALMON, lw=1.5))
ax.annotate("Collider\n(do NOT adjust)", xy=(1, -1), xytext=(2.3, -1),
            fontsize=9, color=SALMON, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=SALMON, lw=1.5))
plt.tight_layout()
plt.show()

### Simulation: Backdoor Adjustment in Action

Let us verify the backdoor criterion with a concrete simulation.  The true data-generating process is:

$$Z \sim \mathcal{N}(0, 1), \qquad X = 0.8\,Z + \varepsilon_X, \qquad Y = \underbrace{0.5}_{\text{true causal effect}}\,X + 0.7\,Z + \varepsilon_Y$$

The true causal effect of $X$ on $Y$ is $\beta_{X \to Y} = 0.5$.  Without adjustment, the naive regression $Y \sim X$ conflates the causal effect with confounding through $Z$.

In [ ]:
# ── Simulation: Backdoor adjustment ──
N = 5000
TRUE_EFFECT = 0.5

Z = rng.normal(0, 1, N)
X = 0.8 * Z + rng.normal(0, 1, N)
Y = TRUE_EFFECT * X + 0.7 * Z + rng.normal(0, 1, N)

# Naive regression: Y ~ X  (biased — ignores confounder Z)
X_naive = sm.add_constant(X)
fit_naive = sm.OLS(Y, X_naive).fit()

# Adjusted regression: Y ~ X + Z  (correct — controls for Z)
X_adj = sm.add_constant(np.column_stack([X, Z]))
fit_adj = sm.OLS(Y, X_adj).fit()

print(f"True causal effect of X on Y: {TRUE_EFFECT}")
print(f"")
print(f"Naive regression (Y ~ X):")
print(f"  Estimated effect: {fit_naive.params[1]:.4f}  "
      f"  95% CI: [{fit_naive.conf_int()[1][0]:.4f}, {fit_naive.conf_int()[1][1]:.4f}]")
print(f"  Bias: {fit_naive.params[1] - TRUE_EFFECT:+.4f}")
print(f"")
print(f"Adjusted regression (Y ~ X + Z):")
print(f"  Estimated effect: {fit_adj.params[1]:.4f}  "
      f"  95% CI: [{fit_adj.conf_int()[1][0]:.4f}, {fit_adj.conf_int()[1][1]:.4f}]")
print(f"  Bias: {fit_adj.params[1] - TRUE_EFFECT:+.4f}")

The naive estimate is biased upward — it attributes part of $Z$'s effect on $Y$ to $X$ (because $Z$ causes both).  The adjusted estimate recovers the true causal effect of 0.5.

This is the backdoor adjustment formula in its linear regression form: the coefficient of $X$ in the regression $Y \sim X + Z$ is the causal effect, provided $\{Z\}$ satisfies the backdoor criterion.

---

## 3. Good Controls

A **good control** is a variable that, when conditioned on, either **removes confounding bias** or **improves precision** without introducing new bias.  There are two main types.

### 3.1 Confounders (common causes)

A variable $Z$ that is a common cause of $X$ and $Y$ — i.e., $Z \to X$ and $Z \to Y$ — is a **confounder**.  Conditioning on a confounder is always beneficial: it blocks the backdoor path $X \leftarrow Z \rightarrow Y$.

$$\text{DAG: } Z \to X, \quad Z \to Y, \quad X \to Y$$

Without adjustment: $E[Y \mid X{=}x]$ mixes the causal effect of $X$ with the spurious association through $Z$.

With adjustment: $E[Y \mid X{=}x, Z{=}z]$ isolates the causal effect, because within each stratum of $Z$, the only remaining association between $X$ and $Y$ is causal.

**We demonstrated this in the simulation above.**

### 3.2 Precision variables (predictors of $Y$ only)

A variable $W$ that affects $Y$ but is independent of $X$ is a **precision variable**:

$$\text{DAG: } X \to Y \leftarrow W$$

Conditioning on $W$ does not reduce confounding (there is no backdoor path involving $W$), but it **reduces the residual variance** of $Y$, thereby narrowing the confidence interval on the causal effect.  This is analogous to including a covariate in a randomised experiment to improve power.

Note: $W$ is *not* a collider in the relevant sense here, because $X$ does not cause $W$.  The arrows are $X \to Y$ and $W \to Y$ — they collide at $Y$, which is the outcome we are modelling, not a variable we condition on in the traditional d-separation sense.

In [ ]:
# ── Precision variable simulation ──
N = 5000
TRUE_EFFECT = 0.5

# W is independent of X, but predicts Y strongly
X_prec = rng.normal(0, 1, N)
W = rng.normal(0, 1, N)
Y_prec = TRUE_EFFECT * X_prec + 2.0 * W + rng.normal(0, 1, N)

# Without W
fit_no_w = sm.OLS(Y_prec, sm.add_constant(X_prec)).fit()

# With W
fit_with_w = sm.OLS(Y_prec, sm.add_constant(np.column_stack([X_prec, W]))).fit()

print(f"True causal effect: {TRUE_EFFECT}")
print(f"")
print(f"Without precision variable W:")
print(f"  Estimate: {fit_no_w.params[1]:.4f},  SE: {fit_no_w.bse[1]:.4f},"
      f"  95% CI width: {(fit_no_w.conf_int()[1][1] - fit_no_w.conf_int()[1][0]):.4f}")
print(f"")
print(f"With precision variable W:")
print(f"  Estimate: {fit_with_w.params[1]:.4f},  SE: {fit_with_w.bse[1]:.4f},"
      f"  95% CI width: {(fit_with_w.conf_int()[1][1] - fit_with_w.conf_int()[1][0]):.4f}")
print(f"")
print(f"Both estimates are unbiased, but including W reduces the SE by "
      f"{(1 - fit_with_w.bse[1]/fit_no_w.bse[1])*100:.0f}%.")

Both estimates are centred on the true effect (no bias), but including $W$ dramatically shrinks the standard error.  This is because $W$ explains a large portion of $Y$'s variance, leaving less residual noise.  In a precision variable, you get a "free lunch": better precision with no cost to validity.

**Summary of good controls:**

| Type | DAG pattern | Effect of conditioning |
|------|-------------|------------------------|
| Confounder | $Z \to X, Z \to Y$ | Removes bias (essential) |
| Precision variable | $W \to Y$, $W \perp\!\!\!\perp X$ | Reduces SE (helpful) |

---

## 4. Bad Controls — Four Types

We now turn to the core of this notebook: four common situations where conditioning on a variable **introduces bias** or **makes things worse**.  For each, we present the DAG, a simulation, and an explanation of the mechanism.

### 4.1 Post-Treatment Variable (Mediator Bias)

**The problem:** $M$ lies on the causal pathway from $X$ to $Y$.  Conditioning on $M$ blocks part of the effect we are trying to measure.

$$\text{DAG: } X \to M \to Y, \quad X \to Y$$

The **total causal effect** of $X$ on $Y$ is:

$$\text{Total effect} = \underbrace{\beta_{X \to Y}}_{\text{direct}} + \underbrace{\beta_{X \to M} \cdot \beta_{M \to Y}}_{\text{indirect via } M}$$

If we condition on $M$, we remove the indirect pathway and recover only the **direct effect** $\beta_{X \to Y}$, not the total effect.  Unless you specifically want the direct effect (which requires additional assumptions), conditioning on a mediator is a mistake.

In [ ]:
# ── Bad control 1: Post-treatment variable (mediator) ──
N = 5000
beta_XY_direct = 0.3     # direct effect X -> Y
beta_XM = 0.8            # X -> M
beta_MY = 0.6            # M -> Y
TRUE_TOTAL = beta_XY_direct + beta_XM * beta_MY  # 0.3 + 0.48 = 0.78

X_m = rng.normal(0, 1, N)
M = beta_XM * X_m + rng.normal(0, 1, N)
Y_m = beta_XY_direct * X_m + beta_MY * M + rng.normal(0, 1, N)

# Correct: Y ~ X  (captures total effect)
fit_total = sm.OLS(Y_m, sm.add_constant(X_m)).fit()

# Bad: Y ~ X + M  (blocks mediated path, captures only direct effect)
fit_direct = sm.OLS(Y_m, sm.add_constant(np.column_stack([X_m, M]))).fit()

print("=== Post-Treatment Variable (Mediator) ===")
print(f"True total effect:  {TRUE_TOTAL:.4f}")
print(f"True direct effect: {beta_XY_direct:.4f}")
print(f"")
print(f"Y ~ X     (correct): {fit_total.params[1]:.4f}  <- estimates total effect")
print(f"Y ~ X + M (bad):     {fit_direct.params[1]:.4f}  <- estimates only direct effect")
print(f"")
print(f"Conditioning on M removed {TRUE_TOTAL - beta_XY_direct:.2f} from the estimate ")
print(f"-- the entire indirect effect through M.")

In [ ]:
# DAG for post-treatment variable
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

G_med = nx.DiGraph()
G_med.add_edges_from([("X", "M"), ("M", "Y"), ("X", "Y")])
pos_med = {"X": (0, 0), "M": (1, 1), "Y": (2, 0)}

draw_dag(G_med, pos_med, ax=axes[0], title="Post-Treatment Variable M")

# Bar chart of estimates
labels = ["True total\neffect", "Y ~ X\n(correct)", "Y ~ X + M\n(bad)"]
values = [TRUE_TOTAL, fit_total.params[1], fit_direct.params[1]]
colors = [PERIWINKLE, EMERALD, SALMON]
axes[1].bar(labels, values, color=colors, edgecolor="black", linewidth=0.8)
axes[1].axhline(TRUE_TOTAL, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[1].set_ylabel("Estimated effect")
axes[1].set_title("Controlling for a mediator understates the total effect")
for i, v in enumerate(values):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

### 4.2 Collider Bias (Endogenous Selection Bias)

**The problem:** $C$ is a **common effect** of $X$ and $Y$ (or of causes of $X$ and $Y$).  Conditioning on a collider **opens** a previously blocked path, creating a spurious association.

$$\text{DAG: } X \to C \leftarrow Y, \quad X \to Y$$

By d-separation, the path $X \to C \leftarrow Y$ is **blocked** when we do not condition on $C$.  But conditioning on $C$ (or on a descendant of $C$) **unblocks** it, inducing a spurious association between $X$ and $Y$.

**Classic example (Berkson's paradox):** Among hospitalised patients (conditioned on hospitalisation, a collider), diseases that are independent in the general population appear negatively correlated.  If you are hospitalised, and you do *not* have disease A, you are more likely to have disease B — because you must have *some* reason to be in the hospital.

**Another example:** The "talent-beauty" correlation.  Among famous people (conditioned on fame), talent and beauty appear negatively correlated — because both can lead to fame, and if you have one, you need less of the other.

In [ ]:
# ── Bad control 2: Collider bias ──
N = 5000
TRUE_EFFECT_C = 0.5

X_c = rng.normal(0, 1, N)
Y_c = TRUE_EFFECT_C * X_c + rng.normal(0, 1, N)
C = 0.6 * X_c + 0.6 * Y_c + rng.normal(0, 1, N)  # collider

# Correct: Y ~ X  (no confounders, no need to adjust)
fit_no_c = sm.OLS(Y_c, sm.add_constant(X_c)).fit()

# Bad: Y ~ X + C  (conditions on collider)
fit_with_c = sm.OLS(Y_c, sm.add_constant(np.column_stack([X_c, C]))).fit()

print("=== Collider Bias ===")
print(f"True causal effect: {TRUE_EFFECT_C:.4f}")
print(f"")
print(f"Y ~ X     (correct):  {fit_no_c.params[1]:.4f}")
print(f"Y ~ X + C (bad):      {fit_with_c.params[1]:.4f}")
print(f"Bias from collider:   {fit_with_c.params[1] - TRUE_EFFECT_C:+.4f}")

In [ ]:
# Visualise collider bias
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

G_col = nx.DiGraph()
G_col.add_edges_from([("X", "Y"), ("X", "C"), ("Y", "C")])
pos_col = {"X": (0, 1), "Y": (2, 1), "C": (1, 0)}
draw_dag(G_col, pos_col, ax=axes[0], title="Collider C",
         highlight_nodes={"C"}, highlight_color=SALMON)

# Scatter: show how conditioning on C distorts the relationship
# Stratify by C: low C vs high C
c_median = np.median(C)
low_c = C < np.percentile(C, 25)
high_c = C > np.percentile(C, 75)

axes[1].scatter(X_c[~low_c & ~high_c], Y_c[~low_c & ~high_c],
                alpha=0.1, s=10, color="grey", label="Middle C")
axes[1].scatter(X_c[low_c], Y_c[low_c], alpha=0.3, s=15,
                color=CYAN, label="Low C (bottom 25%)")
axes[1].scatter(X_c[high_c], Y_c[high_c], alpha=0.3, s=15,
                color=SALMON, label="High C (top 25%)")

# Regression lines within strata
for mask, color, label_suffix in [(low_c, CYAN, "low C"), (high_c, SALMON, "high C")]:
    xx = np.linspace(X_c[mask].min(), X_c[mask].max(), 100)
    fit_stratum = sm.OLS(Y_c[mask], sm.add_constant(X_c[mask])).fit()
    axes[1].plot(xx, fit_stratum.params[0] + fit_stratum.params[1] * xx,
                 color=color, linewidth=2)

# Overall regression line
xx_all = np.linspace(X_c.min(), X_c.max(), 100)
axes[1].plot(xx_all, fit_no_c.params[0] + fit_no_c.params[1] * xx_all,
             color="black", linewidth=2, linestyle="--", label="Overall (correct)")

axes[1].set_xlabel("X")
axes[1].set_ylabel("Y")
axes[1].set_title("Conditioning on C distorts the X-Y relationship")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

Within each stratum of $C$, the slope is attenuated compared to the overall (correct) slope.  This is because knowing $C$ (which depends on both $X$ and $Y$) creates a spurious negative association between $X$ and $Y$: if $C$ is high and $X$ is high, we infer that $Y$ need not be as high, and vice versa.

**Key lesson:** Never condition on a common effect of $X$ and $Y$ (or their descendants).  This is the most counter-intuitive of the bad controls — conditioning on $C$ *feels* like adding information, but it actually **creates** a confound that did not exist.

### 4.3 Precision Parasite

**The problem:** A variable $P$ that is caused by $X$ (and possibly other things) but does not affect $Y$.  Conditioning on $P$ "steals" variance from $X$ without removing any confounding.

$$\text{DAG: } X \to P, \quad X \to Y$$

Since there is no confounding (no backdoor path from $X$ to $Y$), the naive regression $Y \sim X$ already gives the correct causal effect.  Adding $P$ does not help with bias.  But because $P$ is a descendant of $X$, it is correlated with $X$, so including it in the regression partially "explains away" $X$.  This can:

- **Increase** the standard error of $\hat{\beta}_X$ (less precision, not more)
- In the presence of additional confounders, **amplify** their bias

The name "precision parasite" comes from the fact that, unlike a true precision variable (which predicts $Y$ independently of $X$), this variable only predicts $Y$ *through* $X$ — it siphons explanatory power from $X$ without contributing anything new.

In [ ]:
# ── Bad control 3: Precision parasite ──
N = 5000
TRUE_EFFECT_P = 0.5

X_p = rng.normal(0, 1, N)
P = 0.9 * X_p + rng.normal(0, 0.5, N)  # P is caused by X
Y_p = TRUE_EFFECT_P * X_p + rng.normal(0, 1, N)

# Correct: Y ~ X
fit_no_p = sm.OLS(Y_p, sm.add_constant(X_p)).fit()

# Bad: Y ~ X + P  (precision parasite)
fit_with_p = sm.OLS(Y_p, sm.add_constant(np.column_stack([X_p, P]))).fit()

print("=== Precision Parasite ===")
print(f"True causal effect: {TRUE_EFFECT_P:.4f}")
print(f"")
print(f"Y ~ X     (correct): est = {fit_no_p.params[1]:.4f},  SE = {fit_no_p.bse[1]:.4f}")
print(f"Y ~ X + P (bad):     est = {fit_with_p.params[1]:.4f},  SE = {fit_with_p.bse[1]:.4f}")
print(f"")
print(f"Including P increased the SE by {(fit_with_p.bse[1]/fit_no_p.bse[1] - 1)*100:.0f}% "
      f"without reducing bias.")
print(f"P is a parasite: it absorbs variance from X without contributing ")
print(f"any independent information about Y.")

In [ ]:
# DAG for precision parasite
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

G_pp = nx.DiGraph()
G_pp.add_edges_from([("X", "P"), ("X", "Y")])
pos_pp = {"X": (0, 1), "Y": (2, 1), "P": (1, 0)}
draw_dag(G_pp, pos_pp, ax=axes[0], title="Precision Parasite P",
         highlight_nodes={"P"}, highlight_color=SALMON)

# Compare SEs
labels = ["Y ~ X\n(correct)", "Y ~ X + P\n(parasite)"]
ses = [fit_no_p.bse[1], fit_with_p.bse[1]]
colors_bar = [EMERALD, SALMON]
axes[1].bar(labels, ses, color=colors_bar, edgecolor="black", linewidth=0.8)
axes[1].set_ylabel("Standard Error of X coefficient")
axes[1].set_title("Precision parasite inflates the SE")
for i, v in enumerate(ses):
    axes[1].text(i, v + 0.001, f"{v:.4f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

### 4.4 Bias Amplifier

**The problem:** There is an **unobserved confounder** $U$ between $X$ and $Y$.  We observe a variable $A$ that predicts $X$ but does not directly affect $Y$.  Intuitively, $A$ might seem harmless or even helpful (it "explains" $X$).  But conditioning on $A$ actually **amplifies** the bias from the unobserved confounder $U$.

$$\text{DAG: } A \to X, \quad U \to X, \quad U \to Y, \quad X \to Y$$

Here $U$ is unobserved, so we cannot adjust for it.  The question is: does adjusting for $A$ help or hurt?

**Why it amplifies bias:**

Without adjusting for $A$, the variation in $X$ comes from three sources: $A$, $U$, and noise.  The confounding bias is proportional to the fraction of $X$'s variance that comes from $U$.

When we condition on $A$, we "explain away" $A$'s contribution to $X$.  The remaining variation in $X$ is now more heavily loaded with $U$.  So the proportion of $X$-variation due to $U$ *increases*, and the confounding bias gets worse.

Mathematically, in the linear case, the omitted variable bias is:

$$\text{Bias} = \beta_{U \to Y} \cdot \frac{\text{Cov}(X, U \mid \text{adjusted vars})}{\text{Var}(X \mid \text{adjusted vars})}$$

Conditioning on $A$ reduces $\text{Var}(X \mid A)$ (by removing $A$'s contribution) but does not reduce $\text{Cov}(X, U \mid A)$ by as much.  The ratio increases, and the bias increases.

In [ ]:
# ── Bad control 4: Bias amplifier ──
N = 5000
TRUE_EFFECT_A = 0.5

# A -> X, U -> X, U -> Y, X -> Y
A = rng.normal(0, 1, N)
U = rng.normal(0, 1, N)  # unobserved confounder
X_a = 0.8 * A + 0.8 * U + rng.normal(0, 0.5, N)
Y_a = TRUE_EFFECT_A * X_a + 0.7 * U + rng.normal(0, 1, N)

# Cannot adjust for U (unobserved)
# Naive: Y ~ X  (biased due to U)
fit_naive_a = sm.OLS(Y_a, sm.add_constant(X_a)).fit()

# Adjusting for A: Y ~ X + A  (amplifies bias!)
fit_with_a = sm.OLS(Y_a, sm.add_constant(np.column_stack([X_a, A]))).fit()

# Gold standard: Y ~ X + U  (if we could observe U)
fit_gold = sm.OLS(Y_a, sm.add_constant(np.column_stack([X_a, U]))).fit()

print("=== Bias Amplifier ===")
print(f"True causal effect: {TRUE_EFFECT_A:.4f}")
print(f"")
print(f"Y ~ X         (naive):        {fit_naive_a.params[1]:.4f}  "
      f"bias = {fit_naive_a.params[1] - TRUE_EFFECT_A:+.4f}")
print(f"Y ~ X + A     (amplified):    {fit_with_a.params[1]:.4f}  "
      f"bias = {fit_with_a.params[1] - TRUE_EFFECT_A:+.4f}")
print(f"Y ~ X + U     (gold, if U observed): {fit_gold.params[1]:.4f}  "
      f"bias = {fit_gold.params[1] - TRUE_EFFECT_A:+.4f}")
print(f"")
print(f"Adjusting for A made the bias WORSE: from "
      f"{abs(fit_naive_a.params[1] - TRUE_EFFECT_A):.4f} to "
      f"{abs(fit_with_a.params[1] - TRUE_EFFECT_A):.4f}")

In [ ]:
# Visualise bias amplification
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

G_ba = nx.DiGraph()
G_ba.add_edges_from([("A", "X"), ("U", "X"), ("U", "Y"), ("X", "Y")])
pos_ba = {"A": (0, 0), "X": (1, 0), "U": (1.5, 1), "Y": (2, 0)}
draw_dag(G_ba, pos_ba, ax=axes[0], title="Bias Amplifier: A predicts X",
         highlight_nodes={"A", "U"}, highlight_color=SALMON)
# Mark U as unobserved
axes[0].annotate("(unobserved)", xy=(1.5, 1), xytext=(1.5, 1.35),
                 fontsize=9, ha="center", color="grey", fontstyle="italic")

# Bar chart
labels = ["True\neffect", "Y ~ X\n(naive)", "Y ~ X + A\n(amplified)", "Y ~ X + U\n(gold std)"]
values = [TRUE_EFFECT_A, fit_naive_a.params[1], fit_with_a.params[1], fit_gold.params[1]]
colors_bar = [PERIWINKLE, GOLD, SALMON, EMERALD]
axes[1].bar(labels, values, color=colors_bar, edgecolor="black", linewidth=0.8)
axes[1].axhline(TRUE_EFFECT_A, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[1].set_ylabel("Estimated effect")
axes[1].set_title("Bias amplification: adjusting for A makes things worse")
for i, v in enumerate(values):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

The bias amplifier is particularly insidious because $A$ looks like a "good" variable to control for — it predicts $X$ and has no direct effect on $Y$.  In the absence of unobserved confounding, conditioning on $A$ would be harmless.  But in the presence of $U$, it concentrates the remaining variation in $X$ on the confounded component, making the bias worse.

**Summary of bad controls:**

| Type | DAG pattern | Mechanism of harm |
|------|-------------|-------------------|
| Post-treatment (mediator) | $X \to M \to Y$ | Blocks part of causal pathway |
| Collider | $X \to C \leftarrow Y$ | Opens a spurious path |
| Precision parasite | $X \to P$, $P \not\to Y$ | Absorbs $X$-variance, inflates SE |
| Bias amplifier | $A \to X$, $U \to X$, $U \to Y$ | Concentrates $X$-variation on confounded part |

---

## 5. The Table 2 Fallacy

### The problem

A ubiquitous practice in empirical research is to run a multiple regression:

$$Y = \beta_0 + \beta_X X + \beta_{Z_1} Z_1 + \beta_{Z_2} Z_2 + \beta_{Z_3} Z_3 + \varepsilon$$

and then present all coefficients in a table (typically "Table 2" in medical/epidemiological papers), interpreting *each* coefficient as if it were the causal effect of that variable on $Y$.

This is the **Table 2 Fallacy** (Westreich & Greenland, 2013).

### Why it is wrong

The variables $Z_1, Z_2, Z_3$ were included in the regression to **adjust for confounding of the $X \to Y$ relationship**.  They form a valid adjustment set for estimating the causal effect of $X$ on $Y$.  But the very same set of variables is almost certainly **not** the correct adjustment set for estimating the causal effect of, say, $Z_1$ on $Y$.

Each coefficient in a multiple regression has a different causal interpretation that depends on the role of that variable in the DAG:

- $\beta_X$ may be the causal effect of $X$ on $Y$ (if the adjustment set is correct for $X$)
- $\beta_{Z_1}$ is **not** the causal effect of $Z_1$ on $Y$ — it may be biased because the other variables in the model are not the right adjustment set for $Z_1$
- $\beta_{Z_2}$ has yet another interpretation

In general, **each causal question requires its own adjustment set**, derived from the DAG.  A single regression rarely serves multiple causal questions simultaneously.

### Concrete example

Consider the DAG:

$$Z_1 \to X, \quad Z_1 \to Y, \quad X \to Z_2 \to Y, \quad X \to Y$$

- $Z_1$ is a confounder of $X \to Y$. Correct adjustment set for the effect of $X$ on $Y$: $\{Z_1\}$.
- $Z_2$ is a mediator of $X \to Y$.

If we run $Y \sim X + Z_1 + Z_2$:

- $\hat{\beta}_X$ estimates the **direct** effect of $X$ on $Y$ (not the total effect, because we conditioned on the mediator $Z_2$)
- $\hat{\beta}_{Z_1}$ does not estimate the causal effect of $Z_1$ on $Y$, because the adjustment set $\{X, Z_2\}$ is not valid for the causal effect of $Z_1$ (it includes descendants of $Z_1$)
- $\hat{\beta}_{Z_2}$ does not estimate the causal effect of $Z_2$ on $Y$, because we have not adjusted for confounders of $Z_2 \to Y$

In [ ]:
# ── Table 2 Fallacy simulation ──
N = 10000

# True structural model:
# Z1 -> X (coeff 0.8), Z1 -> Y (coeff 0.6)
# X -> Z2 (coeff 0.7), Z2 -> Y (coeff 0.5)
# X -> Y  (direct: 0.4)
beta_Z1_X = 0.8
beta_Z1_Y = 0.6
beta_X_Z2 = 0.7
beta_Z2_Y = 0.5
beta_X_Y_direct = 0.4
beta_X_Y_total = beta_X_Y_direct + beta_X_Z2 * beta_Z2_Y  # 0.4 + 0.35 = 0.75

Z1 = rng.normal(0, 1, N)
X_t2 = beta_Z1_X * Z1 + rng.normal(0, 1, N)
Z2 = beta_X_Z2 * X_t2 + rng.normal(0, 1, N)
Y_t2 = beta_X_Y_direct * X_t2 + beta_Z1_Y * Z1 + beta_Z2_Y * Z2 + rng.normal(0, 1, N)

# The "Table 2" regression: Y ~ X + Z1 + Z2
fit_table2 = sm.OLS(Y_t2, sm.add_constant(np.column_stack([X_t2, Z1, Z2]))).fit()

# Correct regression for causal effect of X on Y: Y ~ X + Z1  (total effect)
fit_correct_x = sm.OLS(Y_t2, sm.add_constant(np.column_stack([X_t2, Z1]))).fit()

# Correct regression for causal effect of Z1 on Y: Y ~ Z1  (no need to adjust for anything)
# Actually Z1's total effect on Y = direct (0.6) + via X -> Y (0.8 * 0.75) = 0.6 + 0.6 = 1.2
beta_Z1_Y_total = beta_Z1_Y + beta_Z1_X * beta_X_Y_total
fit_correct_z1 = sm.OLS(Y_t2, sm.add_constant(Z1)).fit()

print("=== The Table 2 Fallacy ===")
print(f"")
print(f"True causal effects:")
print(f"  X -> Y (total):  {beta_X_Y_total:.4f}")
print(f"  X -> Y (direct): {beta_X_Y_direct:.4f}")
print(f"  Z1 -> Y (total): {beta_Z1_Y_total:.4f}")
print(f"  Z1 -> Y (direct): {beta_Z1_Y:.4f}")
print(f"  Z2 -> Y (direct): {beta_Z2_Y:.4f}")
print(f"")
print(f"Table 2 regression (Y ~ X + Z1 + Z2):")
print(f"  coeff(X)  = {fit_table2.params[1]:.4f}  -- this is the direct effect of X")
print(f"  coeff(Z1) = {fit_table2.params[2]:.4f}  -- NOT the causal effect of Z1")
print(f"  coeff(Z2) = {fit_table2.params[3]:.4f}  -- NOT the causal effect of Z2 on Y")
print(f"")
print(f"Correct regressions for each causal question:")
print(f"  Effect of X on Y (Y ~ X + Z1):  {fit_correct_x.params[1]:.4f}  "
      f"(true total: {beta_X_Y_total:.4f})")
print(f"  Effect of Z1 on Y (Y ~ Z1):     {fit_correct_z1.params[1]:.4f}  "
      f"(true total: {beta_Z1_Y_total:.4f})")

In [ ]:
# DAG for Table 2 Fallacy
fig, ax = plt.subplots(figsize=(7, 4))

G_t2 = nx.DiGraph()
G_t2.add_edges_from([
    ("Z1", "X"), ("Z1", "Y"),
    ("X", "Z2"), ("Z2", "Y"),
    ("X", "Y"),
])
pos_t2 = {"Z1": (0, 1), "X": (1, 1), "Z2": (2, 0), "Y": (3, 1)}
draw_dag(G_t2, pos_t2, ax=ax,
         title="Table 2 Fallacy: each coefficient has a different causal interpretation",
         figsize=(7, 4))

# Annotate roles
ax.annotate("Confounder", xy=(0, 1), xytext=(-0.5, 1.7),
            fontsize=9, color=EMERALD, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=EMERALD, lw=1.5))
ax.annotate("Mediator", xy=(2, 0), xytext=(2.5, -0.6),
            fontsize=9, color=SALMON, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=SALMON, lw=1.5))

plt.tight_layout()
plt.show()

### Lessons from the Table 2 Fallacy

1. **One regression, one causal question.** The coefficient of $X$ in a regression adjusted for the correct confounders estimates the causal effect of $X$ on $Y$.  The coefficients of the control variables do *not* have the same clean interpretation.

2. **Each causal effect requires its own adjustment set.** To estimate the causal effect of $Z_1$ on $Y$, you need to identify the backdoor paths from $Z_1$ to $Y$ and find the correct adjustment set — which is generally different from the adjustment set for $X \to Y$.

3. **Resist the temptation to interpret all coefficients.** In applied papers, every row of the regression table gets a sentence of interpretation.  This is often unjustified.  Report the coefficient of interest (for which you designed the adjustment set) and acknowledge that the other coefficients serve a statistical purpose (confounding control) rather than a causal one.

4. **The DAG makes the problem visible.** Without a causal model, you cannot diagnose the Table 2 Fallacy.  The DAG forces you to think about *why* each variable is in the model and what the coefficient represents.

---

## 6. Automated Adjustment with DoWhy

Drawing DAGs and manually tracing backdoor paths works for small graphs, but becomes unwieldy for complex models with many variables.  **DoWhy** is a Python library that automates the entire causal inference pipeline:

1. **Model:** Specify the causal DAG.
2. **Identify:** Automatically find the estimand (e.g., the backdoor adjustment formula).
3. **Estimate:** Compute the causal effect using the identified adjustment set.
4. **Refute:** Run robustness checks (placebo treatments, random common causes, etc.).

Let us revisit our confounder example from Section 2.

In [ ]:
if HAS_DOWHY:
    import pandas as pd

    # Create a DataFrame from our earlier simulation (Section 2)
    df = pd.DataFrame({"Z": Z, "X": X, "Y": Y})

    # Specify the causal DAG using GML (Graph Modelling Language)
    causal_graph = """
    digraph {
        Z -> X;
        Z -> Y;
        X -> Y;
    }
    """

    # Step 1: Create the causal model
    model = CausalModel(
        data=df,
        treatment="X",
        outcome="Y",
        graph=causal_graph,
    )

    print("=== DoWhy: Causal Model ===")
    print(f"Treatment: X")
    print(f"Outcome: Y")
    print(f"Common causes (potential confounders): {model._common_causes}")
else:
    print("DoWhy not installed — skipping this section.")
    print("Install with: pip install dowhy")

In [ ]:
if HAS_DOWHY:
    # Step 2: Identify the causal effect
    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
    print("=== Identified Estimand ===")
    print(identified_estimand)

In [ ]:
if HAS_DOWHY:
    # Step 3: Estimate the causal effect
    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.linear_regression",
    )
    print("=== Causal Effect Estimate ===")
    print(f"Estimated causal effect (DoWhy): {estimate.value:.4f}")
    print(f"True causal effect:              {TRUE_EFFECT:.4f}")
    print(f"")
    print(f"DoWhy automatically identified Z as a confounder")
    print(f"and used the backdoor adjustment formula.")

In [ ]:
if HAS_DOWHY:
    # Step 4: Refutation — placebo treatment test
    refute_placebo = model.refute_estimate(
        identified_estimand, estimate,
        method_name="placebo_treatment_refuter",
        placebo_type="permute",
    )
    print("=== Refutation: Placebo Treatment ===")
    print(refute_placebo)
    print()
    print("If the effect drops to ~0 with a random (placebo) treatment,")
    print("that supports the causal interpretation of our estimate.")

### What DoWhy does for you

1. **Automatic identification:** Given the DAG, DoWhy finds valid adjustment sets using the backdoor criterion (and other criteria like the front-door criterion when applicable).

2. **Multiple estimation methods:** Beyond linear regression, DoWhy supports propensity score matching, inverse probability weighting, instrumental variables, and more.

3. **Built-in refutation tests:** DoWhy provides several robustness checks:
   - **Placebo treatment:** Replace the real treatment with random noise. The estimated effect should drop to zero.
   - **Random common cause:** Add a random variable as a confounder. The estimate should not change much.
   - **Data subset:** Re-estimate on random subsets. The estimate should be stable.

4. **Transparent assumptions:** By requiring an explicit DAG, DoWhy forces you to state your causal assumptions clearly. The identification step tells you whether your assumptions are sufficient to estimate the causal effect — and if not, why not.

**Limitation:** DoWhy cannot tell you whether your DAG is *correct* — that is a scientific question, not a statistical one.  The tool is only as good as the causal model you provide.

---

## 7. Summary: Decision Flowchart

When deciding whether to include a variable $V$ in your adjustment set for estimating the effect of $X$ on $Y$, ask:

1. **Is $V$ a descendant of $X$?**
   - Yes $\Rightarrow$ Do **not** condition on $V$.  It may be a mediator (blocks the causal path), a collider descendant (opens a spurious path), or a precision parasite.
   - No $\Rightarrow$ Proceed to step 2.

2. **Does $V$ lie on a backdoor path from $X$ to $Y$?**
   - Yes $\Rightarrow$ Conditioning on $V$ may help block this path.  It is a **good control** (confounder).
   - No $\Rightarrow$ Proceed to step 3.

3. **Does $V$ predict $Y$ independently of $X$?**
   - Yes $\Rightarrow$ It is a **precision variable** — helpful for reducing SE, harmless for bias.
   - No $\Rightarrow$ Proceed to step 4.

4. **Does $V$ predict $X$ only?** And is there unobserved confounding?
   - Yes $\Rightarrow$ $V$ may be a **bias amplifier**.  Do not condition on it.
   - No unobserved confounding $\Rightarrow$ Conditioning is harmless but unhelpful.

**When in doubt:** Draw the DAG, identify all paths between $X$ and $Y$, and apply the backdoor criterion formally.  Or use a tool like DoWhy to automate the process.

---

## Exercises

**Exercise 1 (Backdoor criterion — by hand).** Consider the DAG with nodes $\{X, Y, A, B, C, D\}$ and edges:

$$A \to X, \quad A \to B, \quad B \to Y, \quad X \to C, \quad C \to Y, \quad X \to Y, \quad D \to A, \quad D \to Y$$

(a) List all paths from $X$ to $Y$.  Classify each as a causal (directed) path or a backdoor path.

(b) Does $\{A\}$ satisfy the backdoor criterion for $(X, Y)$?  What about $\{B\}$?  What about $\{D\}$?  What about $\{A, D\}$?

(c) Find the **minimal** sufficient adjustment set (the smallest set satisfying the backdoor criterion).

**Exercise 2 (Collider simulation).** Generate data from the following DGP:

$$X \sim \mathcal{N}(0, 1), \quad Y \sim \mathcal{N}(0, 1), \quad C = X + Y + \varepsilon_C$$

where $X$ and $Y$ are independent (there is no causal effect of $X$ on $Y$).  

(a) Verify that the naive regression $Y \sim X$ gives a coefficient near 0 (correct).  
(b) Show that $Y \sim X + C$ gives a spurious negative coefficient on $X$ (collider bias).  
(c) Repeat the simulation 1000 times and plot the sampling distribution of the $X$ coefficient from both regressions.

**Exercise 3 (Bias amplification — varying strength).** Using the bias amplifier DGP from Section 4.4, vary the strength of the $A \to X$ relationship from 0 to 2 (in steps of 0.1) while keeping all other coefficients fixed.  For each value, compute the bias of (i) the naive regression $Y \sim X$ and (ii) the adjusted regression $Y \sim X + A$.  Plot both biases as a function of the $A \to X$ coefficient.  At what point does the amplification become most severe?

**Exercise 4 (Table 2 Fallacy — extended).** Add a second confounder $Z_3$ to the Table 2 example such that $Z_3 \to Z_1$ and $Z_3 \to Y$.  Redraw the DAG.  What is the correct adjustment set for the total causal effect of $X$ on $Y$?  What is the correct adjustment set for the total causal effect of $Z_1$ on $Y$?  Run the regressions and verify.

**Exercise 5 (DoWhy exploration).** Using DoWhy, specify the bias amplifier DAG from Section 4.4 (with $U$ as an unobserved variable).  

(a) What does DoWhy's `identify_effect` return?  Can it identify the causal effect?  
(b) If you *falsely* specify $U$ as observed and include it in the dataset, what happens to the estimate?  
(c) Discuss: what does this exercise tell you about the importance of correct DAG specification?

---

## Key Takeaways

1. **The backdoor criterion** provides a principled, graph-based rule for selecting adjustment variables.  A set $\mathbf{S}$ is valid if (i) no member is a descendant of $X$, and (ii) $\mathbf{S}$ blocks all backdoor paths.

2. **The backdoor adjustment formula** $P(Y \mid \text{do}(X)) = \sum_s P(Y \mid X, S{=}s)\,P(S{=}s)$ connects the interventional distribution to observational quantities, making causal inference from observational data possible.

3. **Good controls:** Confounders (common causes of $X$ and $Y$) remove bias.  Precision variables (independent predictors of $Y$) improve efficiency without affecting bias.

4. **Bad controls can be worse than no controls at all:**
   - **Post-treatment variables** block the causal pathway.
   - **Colliders** open spurious paths.
   - **Precision parasites** inflate standard errors.
   - **Bias amplifiers** worsen confounding from unobserved variables.

5. **The Table 2 Fallacy:** In a multiple regression, each coefficient has a different causal interpretation.  The adjustment set designed for one coefficient is generally invalid for the others.  One regression, one causal question.

6. **Always draw the DAG first.** Without a causal model, you cannot determine which variables to adjust for.  The DAG makes your assumptions explicit and the backdoor criterion makes the logic mechanical.

7. **Tools like DoWhy** can automate identification and estimation once you specify the DAG, but they cannot tell you if the DAG is correct — that requires domain knowledge.

**Next:** [04_do_calculus_intro.ipynb](04_do_calculus_intro.ipynb) — The full do-calculus: three rules for reducing interventional distributions to observational quantities, handling cases where the backdoor criterion fails.

In [ ]:
cfg.save_gifs(clean=True)